# Stable BCE and gradients for manufacturing defects

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

Verify stable BCE and output gradients on local real labels and fixed logits.

In [2]:
def stable_bce_from_logits(logits, targets):
    logits = np.asarray(logits, dtype=np.float64)
    targets = np.asarray(targets, dtype=np.float64)
    if logits.shape != targets.shape or logits.ndim != 1 or logits.size == 0:
        raise ValueError("logits and targets must be equal non-empty vectors")
    if not np.isfinite(logits).all() or not np.isfinite(targets).all():
        raise ValueError("logits and targets must be finite")
    if not np.isin(targets, (0.0, 1.0)).all():
        raise ValueError("targets must be binary")
    losses = np.logaddexp(0.0, logits) - targets * logits
    probabilities = np.empty_like(logits)
    nonnegative = logits >= 0.0
    probabilities[nonnegative] = 1.0 / (1.0 + np.exp(-logits[nonnegative]))
    negative_exponential = np.exp(logits[~nonnegative])
    probabilities[~nonnegative] = negative_exponential / (1.0 + negative_exponential)
    gradients = probabilities - targets
    return losses, gradients, gradients / logits.size

In [3]:
def coordinate_central_difference(objective, values, index, step):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 1 or values.size == 0 or not np.isfinite(values).all():
        raise ValueError("values must be one finite non-empty vector")
    if not 0 <= index < values.size:
        raise IndexError("index is outside the vector")
    if not np.isfinite(step) or step <= 0.0:
        raise ValueError("step must be finite and positive")
    plus = values.copy()
    minus = values.copy()
    plus[index] += step
    minus[index] -= step
    return float((objective(plus) - objective(minus)) / (2.0 * step))

In [4]:
LOCKED_STEPS = tuple(10.0 ** -power for power in range(1, 10))
LOCKED_TOLERANCE = 5e-7


def evaluate_gradient_sweep(kind, targets, outputs, index):
    targets = np.asarray(targets, dtype=np.float64)
    outputs = np.asarray(outputs, dtype=np.float64)
    if kind == "mse":
        objective = lambda values: float(np.mean((values - targets) ** 2))
        analytic = float(2.0 * (outputs[index] - targets[index]) / targets.size)
        differentiable = True
    elif kind == "mae":
        objective = lambda values: float(np.mean(np.abs(values - targets)))
        residual = float(outputs[index] - targets[index])
        analytic = float(np.sign(residual) / targets.size)
        differentiable = residual != 0.0
    elif kind == "bce":
        objective = lambda values: float(
            np.mean(np.logaddexp(0.0, values) - targets * values)
        )
        logit = float(outputs[index])
        if logit >= 0.0:
            probability = float(1.0 / (1.0 + np.exp(-logit)))
        else:
            exponential = float(np.exp(logit))
            probability = exponential / (1.0 + exponential)
        analytic = float((probability - targets[index]) / targets.size)
        differentiable = True
    else:
        raise ValueError(f"unsupported loss kind: {kind}")
    rows = []
    for step in LOCKED_STEPS:
        numerical = coordinate_central_difference(objective, outputs, index, step)
        absolute_error = abs(analytic - numerical)
        rows.append({
            "step": step,
            "analyticValue": analytic,
            "numericalValue": numerical,
            "absoluteError": absolute_error,
            "scaledRelativeError": absolute_error / max(1.0, abs(analytic), abs(numerical)),
            "tolerance": LOCKED_TOLERANCE,
            "differentiable": differentiable,
            "status": (
                "kink"
                if not differentiable
                else "pass" if absolute_error <= LOCKED_TOLERANCE else "fail"
            ),
            "note": (
                "A symmetric difference is not a unique derivative at the MAE kink."
                if not differentiable
                else None
            ),
        })
    return rows

In [5]:
import struct
import zlib


def write_strict_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(
        value,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    ) + "\n"
    path.write_text(payload, encoding="utf-8", newline="\n")


def _png_chunk(kind, payload):
    return (
        struct.pack(">I", len(payload))
        + kind
        + payload
        + struct.pack(">I", zlib.crc32(kind + payload) & 0xFFFFFFFF)
    )


def write_pattern_plot(path, *, title, description, non_color_encoding, series):
    width, height = 960, 540
    pixels = bytearray([248, 250, 252] * width * height)

    def set_pixel(x, y, color):
        if 0 <= x < width and 0 <= y < height:
            offset = (y * width + x) * 3
            pixels[offset : offset + 3] = bytes(color)

    def line(x0, y0, x1, y1, color, dashed=False):
        dx = abs(x1 - x0)
        sx = 1 if x0 < x1 else -1
        dy = -abs(y1 - y0)
        sy = 1 if y0 < y1 else -1
        error = dx + dy
        step = 0
        while True:
            if not dashed or (step // 7) % 2 == 0:
                set_pixel(x0, y0, color)
                set_pixel(x0, y0 + 1, color)
            if x0 == x1 and y0 == y1:
                break
            doubled = 2 * error
            if doubled >= dy:
                error += dy
                x0 += sx
            if doubled <= dx:
                error += dx
                y0 += sy
            step += 1

    left, right, top, bottom = 80, 900, 55, 470
    line(left, bottom, right, bottom, (30, 41, 59))
    line(left, top, left, bottom, (30, 41, 59))
    flattened = [abs(float(value)) for _, values in series for value in values]
    maximum = max(flattened, default=1.0) or 1.0
    colors = ((35, 85, 155), (178, 72, 56), (52, 125, 83), (111, 78, 153))
    for series_index, (_, values) in enumerate(series):
        values = [float(value) for value in values]
        if not values:
            continue
        for index, value in enumerate(values):
            x = left + round(index * (right - left) / max(1, len(values) - 1))
            y = bottom - round(abs(value) / maximum * (bottom - top - 20))
            if index:
                previous = float(values[index - 1])
                previous_x = left + round((index - 1) * (right - left) / max(1, len(values) - 1))
                previous_y = bottom - round(abs(previous) / maximum * (bottom - top - 20))
                line(
                    previous_x,
                    previous_y,
                    x,
                    y,
                    colors[series_index % len(colors)],
                    dashed=series_index % 2 == 1,
                )
            marker = 4 + series_index
            for marker_y in range(y - marker, y + marker + 1):
                for marker_x in range(x - marker, x + marker + 1):
                    if series_index % 2 == 0:
                        draw = abs(marker_x - x) + abs(marker_y - y) <= marker
                    else:
                        draw = (
                            abs(marker_x - x) == marker
                            or abs(marker_y - y) == marker
                        )
                    if draw:
                        set_pixel(
                            marker_x,
                            marker_y,
                            colors[series_index % len(colors)],
                        )

    raw = b"".join(
        b"\x00" + bytes(pixels[row * width * 3 : (row + 1) * width * 3])
        for row in range(height)
    )
    metadata = {
        "Title": title,
        "Description": description,
        "NonColorEncoding": non_color_encoding,
        "Software": "ML Atlas Phase 26 deterministic stdlib PNG writer",
    }
    png = bytearray(b"\x89PNG\r\n\x1a\n")
    png.extend(
        _png_chunk(
            b"IHDR",
            struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0),
        )
    )
    for key, value in metadata.items():
        png.extend(_png_chunk(b"tEXt", key.encode("latin1") + b"\x00" + value.encode("latin1")))
    png.extend(_png_chunk(b"IDAT", zlib.compress(raw, level=9)))
    png.extend(_png_chunk(b"IEND", b""))
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(bytes(png))

In [6]:
DATASET_PATH = Path("../../datasets/loss-functions/secom-manufacturing.csv")
DATASET_MANIFEST_PATH = Path("../../datasets/loss-functions/secom-manufacturing-manifest.json")
SUMMARY_PATH = Path("outputs/bce-gradient-summary.json")
PLOT_PATH = Path("outputs/manufacturing-bce-gradients.png")

In [7]:
frame = pd.read_csv(DATASET_PATH)
dataset_manifest = json.loads(DATASET_MANIFEST_PATH.read_text(encoding="utf-8"))
rows = dataset_manifest["auxiliaryPredictions"]["rows"]
labels = np.asarray([row["label"] for row in rows], dtype=np.float64)
logits = np.asarray([row["logit"] for row in rows], dtype=np.float64)
losses, per_logit_gradients, mean_gradients = stable_bce_from_logits(logits, labels)
if not np.allclose(losses, [row["stableBce"] for row in rows], rtol=0.0, atol=1e-12):
    raise RuntimeError("locked OOF BCE values drifted")
if not np.allclose(
    per_logit_gradients,
    [row["perLogitGradient"] for row in rows],
    rtol=0.0,
    atol=1e-15,
):
    raise RuntimeError("locked OOF gradients drifted")

fixed_probes = []
for logit in (-1000.0, -20.0, 0.0, 20.0, 1000.0):
    for label in (0, 1):
        if logit >= 0.0:
            probability = float(1.0 / (1.0 + np.exp(-logit)))
        else:
            exponential = float(np.exp(logit))
            probability = exponential / (1.0 + exponential)
        with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
            naive_value = float(
                -(label * np.log(probability) + (1 - label) * np.log1p(-probability))
            )
        if np.isfinite(naive_value):
            naive = {"status": "finite", "value": naive_value}
        elif np.isnan(naive_value):
            naive = {"status": "nan", "value": None}
        elif naive_value > 0:
            naive = {"status": "inf", "value": None}
        else:
            naive = {"status": "-inf", "value": None}
        epsilon = 1e-12
        clipped_probability = float(np.clip(probability, epsilon, 1.0 - epsilon))
        clipped_value = float(
            -(label * np.log(clipped_probability)
              + (1 - label) * np.log1p(-clipped_probability))
        )
        stable_value = float(np.logaddexp(0.0, logit) - label * logit)
        fixed_probes.append({
            "source": "synthetic-stability-probe",
            "logit": logit,
            "label": label,
            "probability": probability,
            "naive": naive,
            "clipped": {
                "method": "clipped-probability-bce",
                "status": "finite",
                "value": clipped_value,
                "probability": probability,
                "clippedProbability": clipped_probability,
                "epsilon": epsilon,
                "objectiveChanged": clipped_probability != probability,
            },
            "stable": {
                "method": "stable-logit-bce",
                "status": "finite",
                "value": stable_value,
            },
        })

sweeps = {
    "mse": evaluate_gradient_sweep("mse", [1.0, 2.0], [1.5, 3.0], 1),
    "mae": evaluate_gradient_sweep("mae", [0.0, 2.0], [1.0, 3.0], 0),
    "mae-kink": evaluate_gradient_sweep("mae", [1.0, 2.0], [1.0, 3.0], 0),
    "bce": evaluate_gradient_sweep("bce", [0.0, 1.0], [0.2, -0.4], 1),
}
summary = {
    "contractVersion": "loss-functions-phase-26-output-v1",
    "topicId": "manufacturing-bce-gradients",
    "dataset": {
        "datasetId": dataset_manifest["datasetId"],
        "sha256": dataset_manifest["published"]["sha256"],
        "rowCount": int(labels.size),
        "declaredFeatureCount": dataset_manifest["published"]["declaredFeatureCount"],
        "observedFeatureCount": dataset_manifest["published"]["observedFeatureCount"],
    },
    "aggregate": {
        "meanStableBce": float(np.mean(losses)),
        "rowCount": int(labels.size),
    },
    "rows": rows,
    "highContributionRows": dataset_manifest["auxiliaryPredictions"]["highestContributionRows"],
    "confidentError": dataset_manifest["auxiliaryPredictions"]["confidentError"],
    "fixedProbes": fixed_probes,
    "finiteDifferenceSweeps": sweeps,
    "plot": {
        "path": "outputs/manufacturing-bce-gradients.png",
        "width": 960,
        "height": 540,
        "nonColorEncoding": "Smooth losses use solid circles; MAE kink uses dashed squares",
    },
}
write_strict_json(SUMMARY_PATH, summary)
write_pattern_plot(
    PLOT_PATH,
    title="Manufacturing BCE and gradient verification",
    description="Real SECOM BCE contributions and finite gradient-check errors.",
    non_color_encoding="Smooth losses use solid circle lines; MAE kink uses dashed square line",
    series=(
        ("MSE solid circle", [row["absoluteError"] for row in sweeps["mse"]]),
        ("MAE dashed square", [row["absoluteError"] for row in sweeps["mae-kink"]]),
        ("BCE solid diamond", [row["absoluteError"] for row in sweeps["bce"]]),
    ),
)
print(json.dumps({
    "topicId": summary["topicId"],
    "rowCount": summary["aggregate"]["rowCount"],
    "meanStableBce": summary["aggregate"]["meanStableBce"],
    "confidentErrorStatus": summary["confidentError"]["selectionStatus"],
}, sort_keys=True, allow_nan=False))

{"confidentErrorStatus": "real-oof-row", "meanStableBce": 0.9978726370284953, "rowCount": 1567, "topicId": "manufacturing-bce-gradients"}
